In [ ]:
%%writefile Matrix.cpp
#include "Matrix.h"
#include <iomanip>
#include <stdexcept>
#include <cmath>
#include <vector>
#include <algorithm>

// ---------- Арифметические операции ----------

Matrix Matrix::operator+(const Matrix& b) const {
    if (m != b.m || n != b.n)
        throw std::invalid_argument("Matrix dimensions must agree for +");
    Matrix res(m, n);
    for (int i = 0; i < m * n; ++i)
        res.elems[i] = elems[i] + b.elems[i];
    return res;
}

Matrix Matrix::operator-(const Matrix& b) const {
    if (m != b.m || n != b.n)
        throw std::invalid_argument("Matrix dimensions must agree for -");
    Matrix res(m, n);
    for (int i = 0; i < m * n; ++i)
        res.elems[i] = elems[i] - b.elems[i];
    return res;
}

Matrix Matrix::operator*(const Matrix& b) const {
    if (n != b.m)
        throw std::invalid_argument("Matrix dimensions must agree for *");
    Matrix res(m, b.n);   // по умолчанию заполнена нулями
    for (int i = 0; i < m; ++i)
        for (int k = 0; k < n; ++k) {
            double aik = (*this)[i][k];
            if (aik == 0.0) continue;
            for (int j = 0; j < b.n; ++j)
                res[i][j] += aik * b[k][j];
        }
    return res;
}

// ---------- Метод Гаусса: приведение к ступенчатому виду ----------
// Возвращает ранг матрицы.
int Matrix::gauss(double eps) {
    int i = 0, j = 0;
    while (i < m && j < n) {
        // 1. Поиск максимального по модулю элемента в j-м столбце
        int imax = i;
        double amax = std::fabs((*this)[i][j]);
        for (int k = i + 1; k < m; ++k) {
            double v = std::fabs((*this)[k][j]);
            if (v > amax) { amax = v; imax = k; }
        }
        if (amax <= eps) { ++j; continue; }

        // 2. Обмен строк i и imax (знаки элементов НЕ меняем!)
        if (imax != i) {
            for (int l = j; l < n; ++l)
                std::swap((*this)[i][l], (*this)[imax][l]);
        }

        // 3. Обнуление j-го столбца ниже i-й строки
        for (int k = i + 1; k < m; ++k) {
            double coeff = (*this)[k][j] / (*this)[i][j];
            if (coeff == 0.0) continue;
            for (int l = j; l < n; ++l)
                (*this)[k][l] -= (*this)[i][l] * coeff;
        }
        ++i; ++j;
    }
    return i;   // число использованных ведущих строк = ранг
}

// ---------- Приведение к улучшенному ступенчатому виду (RREF) ----------
int Matrix::rref(double eps) {
    // Сначала приводим к ступенчатому виду
    int rank = gauss(eps);

    // Индексы ведущих столбцов для каждой ненулевой строки
    std::vector<int> pivot(rank, -1);
    for (int i = 0, j = 0; i < rank && j < n; ) {
        if (std::fabs((*this)[i][j]) > eps) {
            pivot[i] = j;
            ++i;
        }
        ++j;
    }

    // Нормируем ведущие строки и обнуляем элементы ВЫШЕ ведущего
    for (int i = rank - 1; i >= 0; --i) {
        int pj = pivot[i];
        // нормировка
        double lead = (*this)[i][pj];
        for (int l = pj; l < n; ++l)
            (*this)[i][l] /= lead;
        // обнуление выше
        for (int k = i - 1; k >= 0; --k) {
            double coeff = (*this)[k][pj];
            if (coeff == 0.0) continue;
            for (int l = pj; l < n; ++l)
                (*this)[k][l] -= (*this)[i][l] * coeff;
        }
    }
    return rank;
}

// ---------- Определитель ----------
double Matrix::determinant(double eps) const {
    if (m != n)
        throw std::invalid_argument("Determinant is defined only for square matrices");
    Matrix a(*this);
    double det = 1.0;
    int sign = 1;

    for (int i = 0, j = 0; i < n && j < n; ++i, ++j) {
        // Поиск ведущего элемента
        int imax = i;
        double amax = std::fabs(a[i][j]);
        for (int k = i + 1; k < n; ++k) {
            double v = std::fabs(a[k][j]);
            if (v > amax) { amax = v; imax = k; }
        }
        if (amax <= eps) return 0.0;    // вырожденная

        if (imax != i) {
            for (int l = 0; l < n; ++l)
                std::swap(a[i][l], a[imax][l]);
            sign = -sign;
        }

        det *= a[i][i];

        for (int k = i + 1; k < n; ++k) {
            double coeff = a[k][j] / a[i][j];
            if (coeff == 0.0) continue;
            for (int l = j; l < n; ++l)
                a[k][l] -= a[i][l] * coeff;
        }
    }
    return sign * det;
}

// ---------- Обратная матрица методом элементарных преобразований ----------
Matrix Matrix::inverse(double eps) const {
    if (m != n)
        throw std::invalid_argument("Inverse is defined only for square matrices");

    int N = n;
    // Расширенная матрица [A | E]
    Matrix aug(N, 2 * N);
    for (int i = 0; i < N; ++i) {
        for (int j = 0; j < N; ++j)
            aug[i][j] = (*this)[i][j];
        aug[i][N + i] = 1.0;
    }

    // Прямой ход с выбором ведущего элемента
    for (int i = 0; i < N; ++i) {
        int imax = i;
        double amax = std::fabs(aug[i][i]);
        for (int k = i + 1; k < N; ++k) {
            double v = std::fabs(aug[k][i]);
            if (v > amax) { amax = v; imax = k; }
        }
        if (amax <= eps)
            throw std::runtime_error("Matrix is singular, inverse does not exist");

        if (imax != i) {
            for (int l = 0; l < 2 * N; ++l)
                std::swap(aug[i][l], aug[imax][l]);
        }

        // Нормировка i-й строки
        double lead = aug[i][i];
        for (int l = 0; l < 2 * N; ++l)
            aug[i][l] /= lead;

        // Обнуление всех остальных строк в i-м столбце
        for (int k = 0; k < N; ++k) {
            if (k == i) continue;
            double coeff = aug[k][i];
            if (coeff == 0.0) continue;
            for (int l = 0; l < 2 * N; ++l)
                aug[k][l] -= aug[i][l] * coeff;
        }
    }

    // Извлекаем правую часть
    Matrix inv(N, N);
    for (int i = 0; i < N; ++i)
        for (int j = 0; j < N; ++j)
            inv[i][j] = aug[i][N + j];
    return inv;
}

// ---------- Решение СЛАУ Ax = b ----------
void Matrix::solveLinearSystem(const double* freeTerms,
                               double* solution,
                               double eps) const {
    if (m != n)
        throw std::invalid_argument("System must be square");
    // x = A^{-1} b
    Matrix inv = inverse(eps);
    for (int i = 0; i < n; ++i) {
        double s = 0.0;
        for (int j = 0; j < n; ++j)
            s += inv[i][j] * freeTerms[j];
        solution[i] = s;
    }
}

// ---------- Ввод/вывод ----------
std::ostream& operator<<(std::ostream& s, const Matrix& a) {
    for (int i = 0; i < a.rows(); ++i) {
        for (int j = 0; j < a.cols(); ++j) {
            if (j > 0) s << " ";
            s << a[i][j];
        }
        s << std::endl;
    }
    return s;
}

std::istream& operator>>(std::istream& s, Matrix& a) {
    for (int i = 0; s.good() && i < a.rows(); ++i)
        for (int j = 0; s.good() && j < a.cols(); ++j)
            s >> a[i][j];
    return s;
}